# Vesuvius Challenge Surface Detection - Training Notebook

**Target: LB Score 0.65+**

Key Improvements:
1. Multiple model architectures (UNet3D, AttentionUNet3D with SE blocks)
2. Advanced loss functions (Dice + CrossEntropy + Focal)
3. Comprehensive 3D data augmentation
4. Deep supervision for better gradients
5. Test-time augmentation (TTA)
6. Optimized post-processing with hysteresis thresholding

## Setup & Imports

In [ ]:
# Install required packages (imagecodecs needed for LZW-compressed TIFF files)
!pip install -q imagecodecs

from IPython.display import clear_output
clear_output()
print("Dependencies installed!")

In [ ]:
import os
import sys
import random
from pathlib import Path
from typing import Optional, Tuple, List, Dict, Any

import numpy as np
import pandas as pd
import tifffile
import zipfile
from tqdm.notebook import tqdm
from scipy.ndimage import gaussian_filter
import scipy.ndimage as ndi
from skimage.morphology import remove_small_objects
from matplotlib import pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Configuration

In [ ]:
# ============================================
# CONFIGURATION - Adjust these as needed
# ============================================

# Paths (Kaggle)
ROOT_DIR = "/kaggle/input/vesuvius-challenge-surface-detection"
TRAIN_IMAGES = f"{ROOT_DIR}/train_images"
TRAIN_LABELS = f"{ROOT_DIR}/train_labels"
TEST_IMAGES = f"{ROOT_DIR}/test_images"
TRAIN_CSV = f"{ROOT_DIR}/train.csv"
TEST_CSV = f"{ROOT_DIR}/test.csv"
OUTPUT_DIR = "/kaggle/working"

# Training settings
SEED = 42
EPOCHS = 50  # Increase for better results (100+ recommended)
BATCH_SIZE = 2  # Reduce if OOM
NUM_WORKERS = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# Model settings
MODEL_TYPE = 'attention_unet3d'  # 'unet3d' or 'attention_unet3d'
NUM_CLASSES = 3
BASE_CHANNELS = 32  # Increase for larger model (48, 64)
CROP_SIZE = (160, 160, 160)
DROPOUT = 0.1
DEEP_SUPERVISION = True
NUM_CROPS_PER_VOLUME = 4

# Inference settings
USE_TTA = True
OVERLAP = 0.5

# Post-processing settings
T_LOW = 0.40
T_HIGH = 0.85
Z_RADIUS = 1
XY_RADIUS = 1
DUST_MIN_SIZE = 150

In [ ]:
# Set seeds for reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 3D Data Augmentation

In [ ]:
class RandomFlip3D:
    """Random flip along each axis with given probability."""
    def __init__(self, p: float = 0.5):
        self.p = p
    
    def __call__(self, image: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        for axis in [0, 1, 2]:
            if random.random() < self.p:
                image = np.flip(image, axis=axis).copy()
                mask = np.flip(mask, axis=axis).copy()
        return image, mask


class RandomRotate3D:
    """Random 90-degree rotations on axial plane."""
    def __init__(self, p: float = 0.5):
        self.p = p
    
    def __call__(self, image: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if random.random() < self.p:
            k = random.randint(1, 3)
            image = np.rot90(image, k=k, axes=(1, 2)).copy()
            mask = np.rot90(mask, k=k, axes=(1, 2)).copy()
        return image, mask


class RandomIntensityShift:
    """Random intensity shift and scale."""
    def __init__(self, shift_range: float = 0.1, scale_range: float = 0.1, p: float = 0.5):
        self.shift_range = shift_range
        self.scale_range = scale_range
        self.p = p
    
    def __call__(self, image: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if random.random() < self.p:
            shift = random.uniform(-self.shift_range, self.shift_range)
            scale = random.uniform(1 - self.scale_range, 1 + self.scale_range)
            image = image * scale + shift
        return image, mask


class RandomGaussianNoise:
    """Add random Gaussian noise."""
    def __init__(self, std_range: Tuple[float, float] = (0.01, 0.05), p: float = 0.3):
        self.std_range = std_range
        self.p = p
    
    def __call__(self, image: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if random.random() < self.p:
            std = random.uniform(*self.std_range)
            noise = np.random.normal(0, std, image.shape).astype(image.dtype)
            image = image + noise
        return image, mask


class RandomCrop3D:
    """Random 3D crop with optional padding."""
    def __init__(self, crop_size: Tuple[int, int, int]):
        self.crop_size = crop_size
    
    def __call__(self, image: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        d, h, w = image.shape
        cd, ch, cw = self.crop_size
        
        # Pad if necessary
        pad_d = max(0, cd - d)
        pad_h = max(0, ch - h)
        pad_w = max(0, cw - w)
        
        if pad_d > 0 or pad_h > 0 or pad_w > 0:
            image = np.pad(image, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='constant')
            mask = np.pad(mask, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='constant')
        
        d, h, w = image.shape
        
        # Random crop
        d_start = random.randint(0, d - cd)
        h_start = random.randint(0, h - ch)
        w_start = random.randint(0, w - cw)
        
        image = image[d_start:d_start+cd, h_start:h_start+ch, w_start:w_start+cw]
        mask = mask[d_start:d_start+cd, h_start:h_start+ch, w_start:w_start+cw]
        
        return image, mask


class Compose3D:
    """Compose multiple transforms."""
    def __init__(self, transforms: List):
        self.transforms = transforms
    
    def __call__(self, image: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        for t in self.transforms:
            image, mask = t(image, mask)
        return image, mask

## Dataset

In [ ]:
class VesuviusDataset(Dataset):
    """Vesuvius Surface Detection Dataset."""
    
    def __init__(
        self,
        image_dir: str,
        label_dir: str,
        csv_file: str,
        transform: Optional[Compose3D] = None,
        crop_size: Tuple[int, int, int] = (160, 160, 160),
        num_classes: int = 3,
        is_train: bool = True,
        num_crops_per_volume: int = 4,
    ):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.transform = transform
        self.crop_size = crop_size
        self.num_classes = num_classes
        self.is_train = is_train
        self.num_crops_per_volume = num_crops_per_volume
        
        # Load CSV
        self.df = pd.read_csv(csv_file)
        all_ids = self.df['id'].tolist()
        
        # Filter to only include IDs where both image and label files exist
        self.image_ids = []
        missing_count = 0
        for img_id in all_ids:
            image_path = self.image_dir / f"{img_id}.tif"
            label_path = self.label_dir / f"{img_id}.tif"
            if image_path.exists() and label_path.exists():
                self.image_ids.append(img_id)
            else:
                missing_count += 1
        
        print(f"Found {len(self.image_ids)} volumes with both image and label")
        if missing_count > 0:
            print(f"Skipped {missing_count} IDs with missing files")
    
    def __len__(self):
        if self.is_train:
            return len(self.image_ids) * self.num_crops_per_volume
        return len(self.image_ids)
    
    def normalize(self, image: np.ndarray) -> np.ndarray:
        """Normalize image using non-zero mean and std."""
        nonzero_mask = image > 0
        if nonzero_mask.any():
            mean = image[nonzero_mask].mean()
            std = image[nonzero_mask].std()
            if std > 0:
                image = (image - mean) / std
        return image
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        volume_idx = idx % len(self.image_ids)
        image_id = self.image_ids[volume_idx]
        
        # Load image
        image_path = self.image_dir / f"{image_id}.tif"
        image = tifffile.imread(str(image_path)).astype(np.float32)
        
        # Load label
        label_path = self.label_dir / f"{image_id}.tif"
        mask = tifffile.imread(str(label_path)).astype(np.int64)
        
        # Normalize image
        image = self.normalize(image)
        
        # Apply transforms
        if self.transform:
            image, mask = self.transform(image, mask)
        
        # Add channel dimension
        image = image[np.newaxis, ...]  # (1, D, H, W)
        
        return {
            'image': torch.from_numpy(image.copy()),
            'mask': torch.from_numpy(mask.copy()),
            'id': image_id
        }

## Loss Functions

In [ ]:
class DiceLoss(nn.Module):
    """Soft Dice Loss for multi-class segmentation."""
    
    def __init__(self, num_classes: int = 3, smooth: float = 1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
    
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        pred = F.softmax(pred, dim=1)
        target_onehot = F.one_hot(target, self.num_classes).permute(0, 4, 1, 2, 3).float()
        
        intersection = (pred * target_onehot).sum(dim=(2, 3, 4))
        union = pred.sum(dim=(2, 3, 4)) + target_onehot.sum(dim=(2, 3, 4))
        
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        
        return 1.0 - dice.mean()


class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance."""
    
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(pred, target, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


class ComboLoss(nn.Module):
    """Combined loss function for better training."""
    
    def __init__(
        self,
        num_classes: int = 3,
        dice_weight: float = 1.0,
        ce_weight: float = 1.0,
        focal_weight: float = 0.5,
    ):
        super().__init__()
        self.dice_loss = DiceLoss(num_classes)
        self.focal_loss = FocalLoss()
        
        self.dice_weight = dice_weight
        self.ce_weight = ce_weight
        self.focal_weight = focal_weight
    
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        loss = 0
        
        if self.dice_weight > 0:
            loss += self.dice_weight * self.dice_loss(pred, target)
        
        if self.ce_weight > 0:
            loss += self.ce_weight * F.cross_entropy(pred, target)
        
        if self.focal_weight > 0:
            loss += self.focal_weight * self.focal_loss(pred, target)
        
        return loss

## 3D UNet Models

In [ ]:
class ConvBlock3D(nn.Module):
    """3D Convolution block with BatchNorm and ReLU."""
    
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout3d(dropout),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class EncoderBlock3D(nn.Module):
    """Encoder block with downsampling."""
    
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.conv = ConvBlock3D(in_channels, out_channels, dropout)
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        features = self.conv(x)
        pooled = self.pool(features)
        return pooled, features


class DecoderBlock3D(nn.Module):
    """Decoder block with upsampling and skip connection."""
    
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.upsample = nn.ConvTranspose3d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = ConvBlock3D(in_channels // 2 + skip_channels, out_channels, dropout)
    
    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.upsample(x)
        
        # Handle size mismatch
        if x.shape != skip.shape:
            diff_d = skip.shape[2] - x.shape[2]
            diff_h = skip.shape[3] - x.shape[3]
            diff_w = skip.shape[4] - x.shape[4]
            x = F.pad(x, [diff_w // 2, diff_w - diff_w // 2,
                         diff_h // 2, diff_h - diff_h // 2,
                         diff_d // 2, diff_d - diff_d // 2])
        
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class UNet3D(nn.Module):
    """3D UNet with deep supervision option."""
    
    def __init__(
        self,
        in_channels: int = 1,
        num_classes: int = 3,
        base_channels: int = 32,
        dropout: float = 0.1,
        deep_supervision: bool = True,
    ):
        super().__init__()
        self.deep_supervision = deep_supervision
        
        # Encoder
        self.enc1 = EncoderBlock3D(in_channels, base_channels, dropout)
        self.enc2 = EncoderBlock3D(base_channels, base_channels * 2, dropout)
        self.enc3 = EncoderBlock3D(base_channels * 2, base_channels * 4, dropout)
        self.enc4 = EncoderBlock3D(base_channels * 4, base_channels * 8, dropout)
        
        # Bottleneck
        self.bottleneck = ConvBlock3D(base_channels * 8, base_channels * 16, dropout)
        
        # Decoder
        self.dec4 = DecoderBlock3D(base_channels * 16, base_channels * 8, base_channels * 8, dropout)
        self.dec3 = DecoderBlock3D(base_channels * 8, base_channels * 4, base_channels * 4, dropout)
        self.dec2 = DecoderBlock3D(base_channels * 4, base_channels * 2, base_channels * 2, dropout)
        self.dec1 = DecoderBlock3D(base_channels * 2, base_channels, base_channels, dropout)
        
        # Output
        self.out = nn.Conv3d(base_channels, num_classes, kernel_size=1)
        
        # Deep supervision heads
        if deep_supervision:
            self.ds4 = nn.Conv3d(base_channels * 8, num_classes, kernel_size=1)
            self.ds3 = nn.Conv3d(base_channels * 4, num_classes, kernel_size=1)
            self.ds2 = nn.Conv3d(base_channels * 2, num_classes, kernel_size=1)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encoder
        x1, skip1 = self.enc1(x)
        x2, skip2 = self.enc2(x1)
        x3, skip3 = self.enc3(x2)
        x4, skip4 = self.enc4(x3)
        
        # Bottleneck
        x = self.bottleneck(x4)
        
        # Decoder
        d4 = self.dec4(x, skip4)
        d3 = self.dec3(d4, skip3)
        d2 = self.dec2(d3, skip2)
        d1 = self.dec1(d2, skip1)
        
        # Output
        out = self.out(d1)
        
        if self.training and self.deep_supervision:
            ds4 = F.interpolate(self.ds4(d4), size=out.shape[2:], mode='trilinear', align_corners=False)
            ds3 = F.interpolate(self.ds3(d3), size=out.shape[2:], mode='trilinear', align_corners=False)
            ds2 = F.interpolate(self.ds2(d2), size=out.shape[2:], mode='trilinear', align_corners=False)
            return out, ds4, ds3, ds2
        
        return out

In [ ]:
class ResBlock3D(nn.Module):
    """Residual block for ResUNet."""
    
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout3d(dropout),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_channels),
        )
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm3d(out_channels),
            )
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu(self.conv(x) + self.shortcut(x))


class SEBlock3D(nn.Module):
    """Squeeze-and-Excitation block for 3D."""
    
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool3d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _, _ = x.shape
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1, 1, 1)
        return x * y.expand_as(x)


class AttentionGate3D(nn.Module):
    """Attention gate for skip connections."""
    
    def __init__(self, gate_channels: int, skip_channels: int, inter_channels: int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv3d(gate_channels, inter_channels, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm3d(inter_channels)
        )
        self.W_x = nn.Sequential(
            nn.Conv3d(skip_channels, inter_channels, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm3d(inter_channels)
        )
        self.psi = nn.Sequential(
            nn.Conv3d(inter_channels, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm3d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, g: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


class AttentionUNet3D(nn.Module):
    """3D Attention UNet with SE blocks."""
    
    def __init__(
        self,
        in_channels: int = 1,
        num_classes: int = 3,
        base_channels: int = 32,
        dropout: float = 0.1,
        deep_supervision: bool = True,
    ):
        super().__init__()
        self.deep_supervision = deep_supervision
        
        # Encoder
        self.enc1 = nn.Sequential(ResBlock3D(in_channels, base_channels, dropout), SEBlock3D(base_channels))
        self.pool1 = nn.MaxPool3d(2)
        
        self.enc2 = nn.Sequential(ResBlock3D(base_channels, base_channels * 2, dropout), SEBlock3D(base_channels * 2))
        self.pool2 = nn.MaxPool3d(2)
        
        self.enc3 = nn.Sequential(ResBlock3D(base_channels * 2, base_channels * 4, dropout), SEBlock3D(base_channels * 4))
        self.pool3 = nn.MaxPool3d(2)
        
        self.enc4 = nn.Sequential(ResBlock3D(base_channels * 4, base_channels * 8, dropout), SEBlock3D(base_channels * 8))
        self.pool4 = nn.MaxPool3d(2)
        
        # Bottleneck
        self.bottleneck = nn.Sequential(ResBlock3D(base_channels * 8, base_channels * 16, dropout), SEBlock3D(base_channels * 16))
        
        # Decoder with attention gates
        self.up4 = nn.ConvTranspose3d(base_channels * 16, base_channels * 8, kernel_size=2, stride=2)
        self.att4 = AttentionGate3D(base_channels * 8, base_channels * 8, base_channels * 4)
        self.dec4 = nn.Sequential(ResBlock3D(base_channels * 16, base_channels * 8, dropout), SEBlock3D(base_channels * 8))
        
        self.up3 = nn.ConvTranspose3d(base_channels * 8, base_channels * 4, kernel_size=2, stride=2)
        self.att3 = AttentionGate3D(base_channels * 4, base_channels * 4, base_channels * 2)
        self.dec3 = nn.Sequential(ResBlock3D(base_channels * 8, base_channels * 4, dropout), SEBlock3D(base_channels * 4))
        
        self.up2 = nn.ConvTranspose3d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.att2 = AttentionGate3D(base_channels * 2, base_channels * 2, base_channels)
        self.dec2 = nn.Sequential(ResBlock3D(base_channels * 4, base_channels * 2, dropout), SEBlock3D(base_channels * 2))
        
        self.up1 = nn.ConvTranspose3d(base_channels * 2, base_channels, kernel_size=2, stride=2)
        self.att1 = AttentionGate3D(base_channels, base_channels, base_channels // 2)
        self.dec1 = nn.Sequential(ResBlock3D(base_channels * 2, base_channels, dropout), SEBlock3D(base_channels))
        
        # Output
        self.out = nn.Conv3d(base_channels, num_classes, kernel_size=1)
        
        # Deep supervision
        if deep_supervision:
            self.ds4 = nn.Conv3d(base_channels * 8, num_classes, kernel_size=1)
            self.ds3 = nn.Conv3d(base_channels * 4, num_classes, kernel_size=1)
            self.ds2 = nn.Conv3d(base_channels * 2, num_classes, kernel_size=1)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        
        # Bottleneck
        b = self.bottleneck(self.pool4(e4))
        
        # Decoder with attention
        d4 = self.up4(b)
        e4 = self.att4(d4, e4)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))
        
        d3 = self.up3(d4)
        e3 = self.att3(d3, e3)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        
        d2 = self.up2(d3)
        e2 = self.att2(d2, e2)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        
        d1 = self.up1(d2)
        e1 = self.att1(d1, e1)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        
        out = self.out(d1)
        
        if self.training and self.deep_supervision:
            ds4 = F.interpolate(self.ds4(d4), size=out.shape[2:], mode='trilinear', align_corners=False)
            ds3 = F.interpolate(self.ds3(d3), size=out.shape[2:], mode='trilinear', align_corners=False)
            ds2 = F.interpolate(self.ds2(d2), size=out.shape[2:], mode='trilinear', align_corners=False)
            return out, ds4, ds3, ds2
        
        return out

## Metrics

In [ ]:
def compute_dice(pred: torch.Tensor, target: torch.Tensor, num_classes: int = 3) -> Dict[str, float]:
    """Compute per-class and mean Dice score."""
    pred_classes = pred.argmax(dim=1)
    dice_scores = {}
    
    for c in range(num_classes):
        pred_c = (pred_classes == c).float()
        target_c = (target == c).float()
        
        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum()
        
        if union > 0:
            dice = (2.0 * intersection / union).item()
        else:
            dice = 1.0 if intersection == 0 else 0.0
        
        dice_scores[f'dice_class_{c}'] = dice
    
    dice_scores['dice_mean'] = np.mean(list(dice_scores.values()))
    return dice_scores

## Training Functions

In [ ]:
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    device: torch.device,
    epoch: int,
    deep_supervision: bool = True,
) -> Dict[str, float]:
    """Train for one epoch."""
    model.train()
    total_loss = 0
    all_dice = []
    
    pbar = tqdm(dataloader, desc=f'Epoch {epoch} - Training')
    
    for batch in pbar:
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)
        
        optimizer.zero_grad()
        
        with autocast('cuda'):
            if deep_supervision:
                outputs = model(images)
                if isinstance(outputs, tuple):
                    main_out, ds4, ds3, ds2 = outputs
                    loss = criterion(main_out, masks)
                    loss += 0.3 * criterion(ds4, masks)
                    loss += 0.2 * criterion(ds3, masks)
                    loss += 0.1 * criterion(ds2, masks)
                else:
                    loss = criterion(outputs, masks)
                    main_out = outputs
            else:
                main_out = model(images)
                loss = criterion(main_out, masks)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
        with torch.no_grad():
            dice = compute_dice(main_out, masks)
            all_dice.append(dice['dice_mean'])
        
        pbar.set_postfix({'loss': loss.item(), 'dice': np.mean(all_dice)})
    
    return {
        'train_loss': total_loss / len(dataloader),
        'train_dice': np.mean(all_dice)
    }


@torch.no_grad()
def validate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    epoch: int,
) -> Dict[str, float]:
    """Validate the model."""
    model.eval()
    total_loss = 0
    all_dice = []
    
    pbar = tqdm(dataloader, desc=f'Epoch {epoch} - Validation')
    
    for batch in pbar:
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)
        
        outputs = model(images)
        if isinstance(outputs, tuple):
            outputs = outputs[0]
        
        loss = criterion(outputs, masks)
        total_loss += loss.item()
        
        dice = compute_dice(outputs, masks)
        all_dice.append(dice['dice_mean'])
        
        pbar.set_postfix({'loss': loss.item(), 'dice': np.mean(all_dice)})
    
    return {
        'val_loss': total_loss / len(dataloader),
        'val_dice': np.mean(all_dice)
    }

## Create Datasets and Model

In [ ]:
# Create transforms
train_transforms = Compose3D([
    RandomCrop3D(CROP_SIZE),
    RandomFlip3D(p=0.5),
    RandomRotate3D(p=0.5),
    RandomIntensityShift(shift_range=0.1, scale_range=0.1, p=0.3),
    RandomGaussianNoise(std_range=(0.01, 0.03), p=0.2),
])

val_transforms = Compose3D([
    RandomCrop3D(CROP_SIZE),
])

# Create datasets
train_dataset = VesuviusDataset(
    image_dir=TRAIN_IMAGES,
    label_dir=TRAIN_LABELS,
    csv_file=TRAIN_CSV,
    transform=train_transforms,
    crop_size=CROP_SIZE,
    num_classes=NUM_CLASSES,
    is_train=True,
    num_crops_per_volume=NUM_CROPS_PER_VOLUME,
)

val_dataset = VesuviusDataset(
    image_dir=TRAIN_IMAGES,
    label_dir=TRAIN_LABELS,
    csv_file=TRAIN_CSV,
    transform=val_transforms,
    crop_size=CROP_SIZE,
    num_classes=NUM_CLASSES,
    is_train=False,
    num_crops_per_volume=1,
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

In [ ]:
# Create model
print(f"Creating model: {MODEL_TYPE}")

if MODEL_TYPE == 'unet3d':
    model = UNet3D(
        in_channels=1,
        num_classes=NUM_CLASSES,
        base_channels=BASE_CHANNELS,
        dropout=DROPOUT,
        deep_supervision=DEEP_SUPERVISION,
    )
elif MODEL_TYPE == 'attention_unet3d':
    model = AttentionUNet3D(
        in_channels=1,
        num_classes=NUM_CLASSES,
        base_channels=BASE_CHANNELS,
        dropout=DROPOUT,
        deep_supervision=DEEP_SUPERVISION,
    )
else:
    raise ValueError(f"Unknown model: {MODEL_TYPE}")

model = model.to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of parameters: {num_params / 1e6:.2f}M")

In [ ]:
# Loss, optimizer, scheduler
criterion = ComboLoss(
    num_classes=NUM_CLASSES,
    dice_weight=1.0,
    ce_weight=1.0,
    focal_weight=0.5,
)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = CosineAnnealingWarmRestarts(
    optimizer,
    T_0=max(EPOCHS // 3, 1),
    T_mult=1,
    eta_min=LEARNING_RATE * 0.01,
)

scaler = GradScaler('cuda')

## Training Loop

In [ ]:
best_dice = 0
history = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Learning rate: {optimizer.param_groups[0]['lr']:.2e}")
    print(f"{'='*60}")
    
    # Train
    train_metrics = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler,
        device, epoch, DEEP_SUPERVISION
    )
    
    # Validate
    val_metrics = validate(model, val_loader, criterion, device, epoch)
    
    # Update scheduler
    scheduler.step()
    
    # Log metrics
    metrics = {**train_metrics, **val_metrics, 'epoch': epoch}
    history.append(metrics)
    
    print(f"\nTrain Loss: {train_metrics['train_loss']:.4f}, Train Dice: {train_metrics['train_dice']:.4f}")
    print(f"Val Loss: {val_metrics['val_loss']:.4f}, Val Dice: {val_metrics['val_dice']:.4f}")
    
    # Save best model
    if val_metrics['val_dice'] > best_dice:
        best_dice = val_metrics['val_dice']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': best_dice,
        }, f'{OUTPUT_DIR}/best_model.pth')
        print(f"Saved new best model with Dice: {best_dice:.4f}")

print(f"\n{'='*60}")
print(f"Training completed! Best validation Dice: {best_dice:.4f}")
print(f"{'='*60}")

## Sliding Window Inference

In [ ]:
class SlidingWindowInference:
    """Sliding window inference with Gaussian weighting for 3D volumes."""
    
    def __init__(
        self,
        model: nn.Module,
        roi_size: Tuple[int, int, int],
        overlap: float = 0.5,
        batch_size: int = 1,
        device: torch.device = None,
    ):
        self.model = model
        self.roi_size = roi_size
        self.overlap = overlap
        self.batch_size = batch_size
        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.importance_map = self._get_importance_map()
    
    def _get_importance_map(self) -> np.ndarray:
        center = [s // 2 for s in self.roi_size]
        sigma = [s / 4 for s in self.roi_size]
        z, y, x = np.ogrid[:self.roi_size[0], :self.roi_size[1], :self.roi_size[2]]
        h = np.exp(-((z - center[0])**2 / (2 * sigma[0]**2) + 
                     (y - center[1])**2 / (2 * sigma[1]**2) + 
                     (x - center[2])**2 / (2 * sigma[2]**2)))
        return h.astype(np.float32)
    
    def __call__(self, volume: np.ndarray, num_classes: int = 3) -> np.ndarray:
        if volume.ndim == 5:
            volume = volume[0, ..., 0]
        elif volume.ndim == 4:
            volume = volume[0] if volume.shape[0] == 1 else volume[..., 0]
        
        d, h, w = volume.shape
        rd, rh, rw = self.roi_size
        
        step_d = int(rd * (1 - self.overlap))
        step_h = int(rh * (1 - self.overlap))
        step_w = int(rw * (1 - self.overlap))
        
        pad_d = max(0, rd - d)
        pad_h = max(0, rh - h)
        pad_w = max(0, rw - w)
        
        if pad_d > 0 or pad_h > 0 or pad_w > 0:
            volume = np.pad(volume, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='constant')
        
        padded_shape = volume.shape
        output_sum = np.zeros((num_classes,) + padded_shape, dtype=np.float32)
        weight_sum = np.zeros(padded_shape, dtype=np.float32)
        
        patches = []
        positions = []
        
        for z in range(0, padded_shape[0] - rd + 1, step_d):
            for y in range(0, padded_shape[1] - rh + 1, step_h):
                for x in range(0, padded_shape[2] - rw + 1, step_w):
                    patch = volume[z:z+rd, y:y+rh, x:x+rw]
                    patches.append(patch)
                    positions.append((z, y, x))
        
        # Add boundary patches
        for z in [max(0, padded_shape[0] - rd)]:
            for y in [max(0, padded_shape[1] - rh)]:
                for x in [max(0, padded_shape[2] - rw)]:
                    if (z, y, x) not in positions:
                        patch = volume[z:z+rd, y:y+rh, x:x+rw]
                        patches.append(patch)
                        positions.append((z, y, x))
        
        self.model.eval()
        with torch.no_grad():
            for i in range(0, len(patches), self.batch_size):
                batch_patches = patches[i:i+self.batch_size]
                batch_positions = positions[i:i+self.batch_size]
                
                batch = np.stack([p[np.newaxis, ...] for p in batch_patches])
                batch_tensor = torch.from_numpy(batch).float().to(self.device)
                
                outputs = self.model(batch_tensor)
                if isinstance(outputs, tuple):
                    outputs = outputs[0]
                
                probs = F.softmax(outputs, dim=1).cpu().numpy()
                
                for j, (z, y, x) in enumerate(batch_positions):
                    output_sum[:, z:z+rd, y:y+rh, x:x+rw] += probs[j] * self.importance_map
                    weight_sum[z:z+rd, y:y+rh, x:x+rw] += self.importance_map
        
        weight_sum = np.maximum(weight_sum, 1e-8)
        output_avg = output_sum / weight_sum[np.newaxis, ...]
        output_avg = output_avg[:, :d, :h, :w]
        
        return output_avg[1]  # Return surface class probability

## Test Time Augmentation

In [ ]:
def predict_with_tta(volume: np.ndarray, swi: SlidingWindowInference, num_classes: int = 3) -> np.ndarray:
    """Predict with test-time augmentation."""
    probs = []
    
    # Original
    probs.append(swi(volume, num_classes))
    
    # Flips
    for axis in [1, 2, 3]:
        img_f = np.flip(volume, axis=axis).copy()
        p = swi(img_f, num_classes)
        actual_axis = axis - 1
        p = np.flip(p, axis=actual_axis).copy()
        probs.append(p)
    
    # Rotations
    for k in [1, 2, 3]:
        img_r = np.rot90(volume.squeeze(), k=k, axes=(1, 2))
        img_r = img_r[np.newaxis, ..., np.newaxis]
        p = swi(img_r, num_classes)
        p = np.rot90(p, k=-k, axes=(1, 2)).copy()
        probs.append(p)
    
    return np.mean(probs, axis=0)

## Post Processing

In [ ]:
def build_anisotropic_struct(z_radius: int, xy_radius: int):
    z, r = z_radius, xy_radius
    if z == 0 and r == 0:
        return None
    if z == 0 and r > 0:
        size = 2 * r + 1
        struct = np.zeros((1, size, size), dtype=bool)
        cy, cx = r, r
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[0, cy + dy, cx + dx] = True
        return struct
    if z > 0 and r == 0:
        struct = np.zeros((2 * z + 1, 1, 1), dtype=bool)
        struct[:, 0, 0] = True
        return struct
    depth = 2 * z + 1
    size = 2 * r + 1
    struct = np.zeros((depth, size, size), dtype=bool)
    cz, cy, cx = z, r, r
    for dz in range(-z, z + 1):
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[cz + dz, cy + dy, cx + dx] = True
    return struct


def topo_postprocess(
    probs: np.ndarray,
    T_low: float = 0.40,
    T_high: float = 0.85,
    z_radius: int = 1,
    xy_radius: int = 1,
    dust_min_size: int = 150,
) -> np.ndarray:
    """Topological post-processing with optimized parameters."""
    strong = probs >= T_high
    weak = probs >= T_low

    if not strong.any():
        return np.zeros_like(probs, dtype=np.uint8)

    struct_hyst = ndi.generate_binary_structure(3, 3)
    mask = ndi.binary_propagation(strong, mask=weak, structure=struct_hyst)

    if not mask.any():
        return np.zeros_like(probs, dtype=np.uint8)

    if z_radius > 0 or xy_radius > 0:
        struct_close = build_anisotropic_struct(z_radius, xy_radius)
        if struct_close is not None:
            mask = ndi.binary_closing(mask, structure=struct_close)
    
    if dust_min_size > 0:
        mask = remove_small_objects(mask.astype(bool), min_size=dust_min_size)

    return mask.astype(np.uint8)

## Inference and Submission

In [ ]:
# Load best model
checkpoint = torch.load(f'{OUTPUT_DIR}/best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded model with validation Dice: {checkpoint['val_dice']:.4f}")

# Create sliding window inference
swi = SlidingWindowInference(
    model,
    roi_size=CROP_SIZE,
    overlap=OVERLAP,
    batch_size=1,
    device=device,
)

In [ ]:
def normalize_volume(image: np.ndarray) -> np.ndarray:
    nonzero_mask = image > 0
    if nonzero_mask.any():
        mean = image[nonzero_mask].mean()
        std = image[nonzero_mask].std()
        if std > 0:
            image = (image - mean) / std
    return image


def load_volume(path: str) -> np.ndarray:
    vol = tifffile.imread(path)
    vol = vol.astype(np.float32)
    vol = normalize_volume(vol)
    vol = vol[np.newaxis, ..., np.newaxis]
    return vol

In [ ]:
# Load test data
test_df = pd.read_csv(TEST_CSV)
print(f"Test samples: {len(test_df)}")

# Create submission
output_masks_dir = f"{OUTPUT_DIR}/submission_masks"
os.makedirs(output_masks_dir, exist_ok=True)

zip_path = f"{OUTPUT_DIR}/submission.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Inference'):
        image_id = row['id']
        tif_path = f"{TEST_IMAGES}/{image_id}.tif"
        
        # Load and preprocess
        volume = load_volume(tif_path)
        
        # Inference with TTA
        if USE_TTA:
            probs = predict_with_tta(volume, swi, NUM_CLASSES)
        else:
            probs = swi(volume, NUM_CLASSES)
        
        # Post-processing
        mask = topo_postprocess(
            probs,
            T_low=T_LOW,
            T_high=T_HIGH,
            z_radius=Z_RADIUS,
            xy_radius=XY_RADIUS,
            dust_min_size=DUST_MIN_SIZE,
        )
        
        # Save mask
        out_path = f"{output_masks_dir}/{image_id}.tif"
        tifffile.imwrite(out_path, mask.astype(np.uint8))
        
        # Add to zip
        z.write(out_path, arcname=f"{image_id}.tif")
        os.remove(out_path)

print(f"\nSubmission saved to: {zip_path}")

## Visualize Sample

In [ ]:
# Visualize a sample prediction
sample_id = test_df['id'].iloc[0]
sample_path = f"{TEST_IMAGES}/{sample_id}.tif"
sample_vol = load_volume(sample_path)

if USE_TTA:
    sample_probs = predict_with_tta(sample_vol, swi, NUM_CLASSES)
else:
    sample_probs = swi(sample_vol, NUM_CLASSES)

sample_mask = topo_postprocess(
    sample_probs,
    T_low=T_LOW,
    T_high=T_HIGH,
    z_radius=Z_RADIUS,
    xy_radius=XY_RADIUS,
    dust_min_size=DUST_MIN_SIZE,
)

# Plot
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
D = sample_vol.squeeze().shape[0]
slices = np.linspace(0, D-1, 5).astype(int)

for i, s in enumerate(slices):
    axes[0, i].imshow(sample_vol.squeeze()[s], cmap='gray')
    axes[0, i].set_title(f'Slice {s}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(sample_mask[s], cmap='hot')
    axes[1, i].set_title(f'Prediction {s}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## Done!

Your submission is ready at `/kaggle/working/submission.zip`